# Test Report - Dedicated Position (for Integrated Game)

## Preprocess

### Packages and Constants

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import datetime
from pyquaternion import Quaternion
import plotly.express as px
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
import argparse

In [ ]:
config_list = [
  'startedTime',
  'user',
  'stage',
  'beginScore',
  'mode',
  'fov',
  'method',
  'interaxial',
  'sensitivity',
  'frameLength',
]

result={
  'method': 'na',
  'stage': 'na',
  'user': 'na',
  'score': -1,
  'widePortion': -1,
  'transitionPortion': -1,
  'transitionCount': -1,
  'head_qu_distance': -1,
  'head_qu_mean_velocity': -1,
  'head_qu_mean_acc': -1,
  'rightHand_qu_distance': -1,
  'rightHand_qu_mean_velocity': -1,
  'rightHand_qu_mean_acc': -1,
  'leftHand_qu_distance': -1,
  'leftHand_qu_mean_velocity': -1,
  'leftHand_qu_mean_acc': -1,
  'player_mean_velocity': -1,
  'player_mean_acc': -1,
}

### Only For Packaging

In [ ]:
# parser = argparse.ArgumentParser(description="pass file path here")
# parser.add_argument('--file', type=str, help='e.g. ../data/chi24/formal/P001/lorem.txt')
# args = parser.parse_args()

# Access the arguments using args.file

In [ ]:
# !jupyter nbconvert --to script report_game.ipynb

### Utilities

In [ ]:
############### General ###############
def remove_outlier (arr):
  mean = np.mean(arr, axis=0)
  std = np.std(arr, axis=0)
  final_list = [x for x in arr if (x > mean - 2 * std)]
  final_list = [x for x in final_list if (x < mean + 2 * std)]
  return final_list

############### Euler ###############
def euler_diff (arr, time):
  diff = []
  for i in range(len(arr)-1):
    diff.append((np.array(arr[i+1])-np.array(arr[i]))/time)
  return diff

def euler_diff_cir (arr, time):
  diff = []
  for i in range(len(arr)-1):
    diff.append(euler_diff_cir_array(arr[i], arr[i+1])/time)
  return diff
  
def euler_diff_cir_array (arr1, arr2):
  res = np.array([0,0,0])
  if (len(arr1) != len(arr2)):
    print("Error: length of two arrays are not equal")
    return
  for i in range(len(arr1)):
    res[i] = euler_diff_cir_element(arr1[i], arr2[i])
  return res

def euler_diff_cir_element (a, b):
  if (b - a > 180):
    return (b-360)-a
  elif (b - a < -180):
    return b-(a-360)
  else:
    return b-a

def euler_distance (arr, threshold = 0):
  sum = 0
  for i in range(len(arr)-1):
    if abs(arr[i+1]-arr[i]) > threshold:
      sum += abs(arr[i+1]-arr[i])
  return sum

def euler_circle (arr):
  circle = []
  for i in range(len(arr)):
    if (arr[i] > 180):
      circle.append(arr[i]-360)
    else:
      circle.append(arr[i])
  return circle

def euler_negative (arr):
  neg = []
  for i in range(len(arr)):
    neg.append(-arr[i])
  return neg

############### Quaternion ###############
def quaternion_distance (arr, threshold = 0):
  sum = 0
  for i in range(len(arr)-1):
    if abs(Quaternion.absolute_distance(Quaternion(arr[i]), Quaternion(arr[i+1]))) > threshold:
      sum += Quaternion.absolute_distance(Quaternion(arr[i]), Quaternion(arr[i+1]))
  return sum

def quaternion_diff_abs (arr, time):
  diff = []
  for i in range(len(arr)-1):
    diff.append(Quaternion.absolute_distance(Quaternion(arr[i]), Quaternion(arr[i+1]))/time)
  return diff

def quaternion_diff (arr, time):
  diff = []
  for i in range(len(arr)-1):
    diff.append(Quaternion.distance(Quaternion(arr[i]), Quaternion(arr[i+1]))/time)
  return diff

### Filename

In [ ]:
userList = [
  'P008',
  'P017',
  'P036',
  'P034',
  'P005',
  'P019',
  'P015',
  'P031',
  'P018',
  'P025',
  'P006',
  'P030',
  'P014',
  'P010'
]

methodList = [
  'Static',
  'Dynamic'
]

In [ ]:
userIdx = 0
methodIdx = 1

In [ ]:
positionSet = []

In [ ]:
for userIdx in range(len(userList)):
  # path = "../data/"
  path = "../data/chi24/summary/"
  filename = "LOG_Sum_" + userList[userIdx] + "_" + methodList[methodIdx] + ".txt"

  print("⛳️ processing: " + userList[userIdx])

  config = {}
  headAngle_qu = np.array([[0,0,0,0]])
  headAngle_eu = np.array([[0,0,0]])
  rightHandPosition = np.array([[0,0,0]])
  leftHandPosition = np.array([[0,0,0]])
  rightHandAngle_qu = np.array([[0,0,0,0]])
  leftHandAngle_qu = np.array([[0,0,0,0]])
  rightHandAngle_eu = np.array([[0,0,0]])
  leftHandAngle_eu = np.array([[0,0,0]])
  position = np.array([[0,0,0]])
  unstruct_tags = []
  unstruct_zoom_tags = []
  duration = 0.0

  # completeFileName = args.file
  completeFileName = path + filename

  f = open(completeFileName, 'r')
  count = 0
  frameCount = 0
  for i, line in enumerate(f):
    if i < len(config_list) and i == count:
      if (line.find("startedTime") == 0):
        startedTime = datetime.datetime.strptime(line[line.find(":")+1:].strip('\n'), '%m/%d/%Y %I:%M:%S %p')
    if i >= len(config_list) and i == count:
      if (line.find("endedTime") == 0):
        endedTime = datetime.datetime.strptime(line[line.find(":")+1:].strip('\n'), '%m/%d/%Y %I:%M:%S %p')
      if (line.find("*") != 0 and line.find("#") != 0):
        frameCount += 1
    count += 1

  # print("startedTime: ", startedTime)
  # print("endedTime: ", endedTime)
  actualDuration = (endedTime - startedTime).total_seconds()
  # print("frameCount: ", frameCount)
  # print("actual duration: ", actualDuration, "seconds")
  actualFrameRate = frameCount / actualDuration # in FPS
  # print("actual frameRate: ", actualFrameRate, "FPS")
  turncatePoint = round(240 * actualFrameRate)
  # print("trucate point frame: ", turncatePoint)

  f = open(completeFileName, 'r')
  count = 0
  frameCount = 0
  for i, line in enumerate(f):
    if (frameCount > turncatePoint):
      break
    if (line == "===\n"):
      break
    if i < len(config_list) and i == count:
      config[config_list[count]] = line[line.find(":")+1:].strip('\n')
    if i >= len(config_list) and i == count:
      if (line.find("*") == 0):
        unstruct_tags.append(str(frameCount) + line.split("\n")[0])
      elif (line.find("#") == 0):
        unstruct_zoom_tags.append(str(frameCount) + line.split("\n")[0])
      else:
        frameCount += 1
        headAngle_qu = np.append(headAngle_qu, [[ float(line.split("%")[0][1:].strip(')').split(",")[0]), 
                                                  float(line.split("%")[0][1:].strip(')').split(",")[1]), 
                                                  float(line.split("%")[0][1:].strip(')').split(",")[2]),
                                                  float(line.split("%")[0][1:].strip(')').split(",")[3]) ]], 
                                                  axis=0)
        
        headAngle_eu = np.append(headAngle_eu, [[ float(line.split("%")[1][1:].strip(')').split(",")[0]), 
                                                  float(line.split("%")[1][1:].strip(')').split(",")[1]), 
                                                  float(line.split("%")[1][1:].strip(')').split(",")[2]) ]], 
                                                  axis=0)

        rightHandPosition = np.append(rightHandPosition, [[ float(line.split("%")[2][1:].strip(')').split(",")[0]), 
                                                            float(line.split("%")[2][1:].strip(')').split(",")[1]), 
                                                            float(line.split("%")[2][1:].strip(')').split(",")[2]) ]], 
                                                            axis=0)
        
        leftHandPosition = np.append(leftHandPosition, [[ float(line.split("%")[3][1:].strip(')').split(",")[0]), 
                                                          float(line.split("%")[3][1:].strip(')').split(",")[1]), 
                                                          float(line.split("%")[3][1:].strip(')').split(",")[2]) ]], 
                                                          axis=0)
        
        rightHandAngle_qu = np.append(rightHandAngle_qu, [[ float(line.split("%")[4][1:].strip(')').split(",")[0]), 
                                                            float(line.split("%")[4][1:].strip(')').split(",")[1]), 
                                                            float(line.split("%")[4][1:].strip(')').split(",")[2]),
                                                            float(line.split("%")[4][1:].strip(')').split(",")[3]) ]], 
                                                            axis=0)
              
        leftHandAngle_qu = np.append(leftHandAngle_qu, [[ float(line.split("%")[5][1:].strip(')').split(",")[0]), 
                                                          float(line.split("%")[5][1:].strip(')').split(",")[1]), 
                                                          float(line.split("%")[5][1:].strip(')').split(",")[2]),
                                                          float(line.split("%")[5][1:].strip(')').split(",")[3]) ]], 
                                                          axis=0)
        
        rightHandAngle_eu = np.append(rightHandAngle_eu, [[ float(line.split("%")[6][1:].strip(')').split(",")[0]), 
                                                            float(line.split("%")[6][1:].strip(')').split(",")[1]), 
                                                            float(line.split("%")[6][1:].strip(')').split(",")[2]) ]], 
                                                            axis=0)
              
        leftHandAngle_eu = np.append(leftHandAngle_eu, [[ float(line.split("%")[7][1:].strip(')').split(",")[0]), 
                                                          float(line.split("%")[7][1:].strip(')').split(",")[1]), 
                                                          float(line.split("%")[7][1:].strip(')').split(",")[2]) ]], 
                                                          axis=0)
        
        position = np.append(position, [[ float(line.split("%")[8][1:].strip(')\n').split(",")[0]), 
                                          float(line.split("%")[8][1:].strip(')\n').split(",")[1]), 
                                          float(line.split("%")[8][1:].strip(')\n').split(",")[2]) ]], 
                                          axis=0)
    count += 1

  # clear initialized zeros
  headAngle_qu = headAngle_qu[1:]
  headAngle_eu = headAngle_eu[1:]
  rightHandPosition = rightHandPosition[1:]
  leftHandPosition = leftHandPosition[1:]
  rightHandAngle_qu = rightHandAngle_qu[1:]
  leftHandAngle_qu = leftHandAngle_qu[1:]
  rightHandAngle_eu = rightHandAngle_eu[1:]
  leftHandAngle_eu = leftHandAngle_eu[1:]
  position = position[1:]
  duration = (1/actualFrameRate) * frameCount # trucated duration

  # check fov
  if (config['mode'] == 'Static'):
    config.update({'method': 'N/A'})
    config.update({'sensitivity': 'N/A'})
    if (config['fov'] != 'Normal'):
      print("\n⚠️ STATIC FOV IS NOT NORMAL")
  elif (config['mode'] == 'Dynamic'):
    config.update({'fov': 'N/A'})

  # check duration
  config.update({'frameLength': (1/actualFrameRate)})
  config.update({'duration': duration}) # trucated duration

  # check score
  if (config['stage'] == '3 maze'):
    config.update({'score': duration})
  else:
    config.update({'score': len(unstruct_tags)})
  if (config['beginScore'] != '0 Pt'):
    print("\n⚠️ BEGIN SCORE IS NOT 0")

  # print("\nconfig", config)
  # print("\ndocument length", headAngle_qu.shape[0])
  # print("\nhead angle quaternion", headAngle_qu)
  # print("\nhead angle euler", headAngle_eu)
  # print("\nright hand pos", rightHandPosition)
  # print("\nleft hand pos", leftHandPosition)
  # print("\nright hand quaternion", rightHandAngle_qu)
  # print("\nleft hand quaternion", leftHandAngle_qu)
  # print("\nright hand ang", rightHandAngle_eu)
  # print("\nleft hand ang", leftHandAngle_eu)
  # print("\nposition", position)

  ### totlly 9 columns
  
  positionSet.insert(len(positionSet), position)

In [ ]:
positionSet

## Report

#### Position

In [ ]:
position_x = [arr[:, 0] for arr in positionSet]
position_y = [arr[:, 2] for arr in positionSet]

In [ ]:
# calculate each subject's standard deviation
position_x_std = [np.std(arr) for arr in position_x]
position_y_std = [np.std(arr) for arr in position_y]

print("position_x_std", position_x_std)
print("position_y_std", position_y_std)

# calculate total mean
position_x_std_mean = np.mean(position_x_std, axis=0)
position_y_std_mean = np.mean(position_y_std, axis=0)

print("position_x_mean", position_x_std_mean)
print("position_y_mean", position_y_std_mean)

##### Player Displacement

In [ ]:
def calculatePlayerDisplacement(positionSet):
  playerDisplacement = []
  for i in range(len(positionSet)):
    playerDisplacement.append(np.linalg.norm(positionSet[i][-1] - positionSet[i][0]))
  return playerDisplacement

In [ ]:
player_displacement = calculatePlayerDisplacement(positionSet)
print(player_displacement)

# calculate whole set average
player_displacement_avg = np.mean(player_displacement)
print("player displacement average in", config['mode'], "is", player_displacement_avg)

# calculate the std
player_displacement_std = np.std(player_displacement)
print("player displacement std in", config['mode'], "is", player_displacement_std)

##### Player Position Overlay

In [ ]:
size = (12,12)

fig, ax = plt.subplots(figsize=size)
fig.subplots_adjust(bottom=0.2, left=0.2)

for ps in positionSet:
  plt.scatter(ps[:,0], ps[:,2], c='r', alpha=0.025, marker='.')

# plt.scatter(position[:,0], position[:,2], c='b', alpha=0.05, marker='.')
# 0:x 2:y 0,2 are in the top view surface

if (config['stage'].split(' ')[0] == '3'):
  ext = [-37, 4, 33, 74.5]
  img = plt.imread("Assets/maze.png")

if (config['stage'].split(' ')[0] == '4'):
  ext = [-11, 48, -6.5, 15.5]
  img = plt.imread("Assets/game.png")

if (config['stage'].split(' ')[0] == '4' or config['stage'].split(' ')[0] == '3'):
  plt.xlim(ext[0], ext[1])
  plt.ylim(ext[2], ext[3])
  plt.imshow(img, zorder=0, extent=ext)
  aspect=img.shape[0]/float(img.shape[1])*((ext[1]-ext[0])/(ext[3]-ext[2]))
  plt.gca().set_aspect(aspect)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title(config['stage'].split(' ')[1] + ' stage: Player Position Distribution in ' + config['mode'] + ' Mode')

plt.show()